# Purpose

The document type attachment_and_transfer_order is higly confused with pfub_erlass by llms.
The purpose for this notebook is to check keyword differences between this two document type so I can adjust the prompt

In [11]:
import pandas as pd
from collections import Counter
import re
import sys, os

In [2]:
WRONG_ATTACHMENT_IDS = [
    "dcf3e789-6aa7-595d-94c7-9133ae2b2375",
    "4b468040-5e4c-56c9-9a7c-83e39555ce59",
    "f05cbba0-d4b6-5f96-8d51-dddb8711cfef",
    "79bdb5e3-e2f8-5952-a870-866dc32f0fa2",
    "1783cc71-6dd7-5a42-b96f-a8fdc46240cd",
    "f93f1468-3e58-5268-be16-804f969609df",
    "9998db95-f3ff-5901-a853-463457eafc99",
    "ecc0819b-3eb5-57a5-b30f-cc860fde3260",
    "745cdb39-5a3f-5f17-a639-4df398811282",
    "7251b0ad-98a8-586f-b910-b72172f6a675",
    "162e66d5-2bec-5c50-a1a7-9ae9e35fba06",
    "4181f6f9-45e4-5891-864e-bea13c19eeed",
    "fc30ab75-3583-55a9-867c-8183aa0f1da0",
    "bbee808b-ae0d-5c0e-8b24-1f614c79cc35",
    "7197d1c5-5282-5bf5-b826-7c693be462ed",
    "5918518c-3a7f-5149-84a7-fc008755665f",
    "4fd5f966-8759-5e8c-ae00-9665ef0f719f",
    "14137bb3-d5ad-5caa-bd04-2513d04b7156",
    "e9546138-cc42-512c-badf-2191fe7c6845",
    "ba9b2867-54c4-5a1a-957a-df85ccb12858",
    "7c5d973e-aec8-5d35-a306-5729de084729",
    "ba743fe8-c65c-5596-93e0-913e42cfbdb7",
    "e747461d-a3b6-5eb1-92e1-a11c4ca72d87",
    "f4620256-d5f6-5dec-b73e-98c26329bd22",
    "be8836e7-8557-54a6-a52f-c252ee4c1ff3",
    "96035b43-7b34-5c41-a7ae-0799d634e6dd",
    "528cfb27-b6e2-5ff4-b12c-030e84979afb",
    "e9b9507f-518b-59cf-9a30-d557205315af",
    "7949a855-19c7-5beb-a98e-57f09972770b",
    "9cc7ec33-252f-55c8-8687-54a0b77b12a0",
    "6c5d67b1-539f-5444-9af1-3dad710f760c",
    "73c8d0e8-4602-5c6a-bbc3-ed8d3b1a288b",
    "cfa79437-4913-5e4b-a221-1b1670cb5167",
    "8e951b79-de8d-56d0-90c6-6cdbf2f53d1f",
    "f82a610e-684c-5b68-b672-a12ad5d57b39",
    "8ea49170-8b08-5967-9eec-f975c9b6e6e3",
    "cb62e6fb-bba2-5d37-b362-a69309a7a945",
    "dabc5a9c-4804-51aa-9a6a-31c80b70deaa",
    "52bc4e87-58e3-57cd-b27e-c2c7995a80bb",
    "5a8001a0-169a-50a7-b47b-1e7b8afd4b19",
    "bdcf3382-b90d-584b-bab8-8a3f46deefd9",
    "2511083c-839b-5e6e-9c37-59de169f98c1",
    "d49d01da-4695-57bd-bfcf-2053ffac080b",
    "67605b1e-1568-5dda-9ddb-ea01d3919931",
    "15d1e0ba-b68e-5aed-824a-fdc555b78e5e",
    "73e813d0-7dd7-584c-8395-0a9ea1b73a80",
    "f5e746c1-7fe8-55b5-b88c-3cd7622ce605",
    "1e9baa4b-78e4-50cf-9e6f-b02fddc4f238",
    "8ce044ba-cb50-562e-9a6a-fbba9507b39a",
    "7ef16f27-53e0-541f-adf1-2d4a89d68705",
    "5a09ec97-0bf5-5ed9-a6d1-9a943a2430f8",
    "1b129c26-1022-5b8a-b01f-0d1c43070308",
    "ccf89b49-9604-5e9f-a785-2142a250a8e4",
    "88a8067d-29cc-52e5-9537-cc28c2df30b3",
    "85746bc5-0d22-5bc0-afbf-2f143bd17eb1",
    "f87cb772-214e-53c1-90c7-98814d8ebd52",
    "65629123-6d31-5c6b-b11b-31f7ae2993b7",
    "e3b8f42c-43ae-5d00-b377-37fbc2d7a746",
    "5641ea07-8646-5146-88e3-6277cd609fb9",
    "0c49b2d5-5441-555e-ae8e-d93d7fcf7a31",
    "26ad1531-bc99-5a73-a730-4327ecb746f1",
    "24a243ea-90da-5813-a9f8-55f7c6ee38b8",
    "bb071e83-ba9a-5b4b-9afd-8bd7113d75b1",
    "05c62246-c1ca-55f8-8e32-2a7daab5c11c",
    "09084df7-377a-56a5-9c09-553588f6175c",
    "1a224b0e-c138-592a-8d16-9d70a04b6159",
    "a7f979e2-aaa9-5903-b078-83b24fbf5d25",
    "992619d8-fc0b-511a-8744-dacf6a5c18dc",
    "309dba0c-01a8-5375-b573-4d9b9608fe18",
    "2cebfd35-df33-5f2c-a3cc-8f9c05287b4d",
    "bc6fee94-62f2-5283-9f2a-26aedf299bbf",
    "ede98573-b85a-52bc-9430-79a7599ebea8",
    "12710113-77e5-53aa-a593-490894527979",
    "ee18240e-d76c-51d1-8d3f-afaa0753c13a",
    "ed9c292c-d18a-5366-a890-7564a579cb5a",
    "0ce56362-2b17-532d-81b2-e782bb9d239f",
    "2e37295a-b3f0-5b6f-afe8-8d94e7e9d174",
    "e205ef3e-3956-5038-a798-2313a00b8193",
    "c1c13f2c-a21b-5b86-af8d-0af5b136e9ba",
    "585c6ddd-d1d3-50fb-bcbb-c45ba2900797",
    "fed1a491-10c0-5606-94f6-61e1dcd17c38",
    "09fc34df-5423-5f4f-9613-8e5489f24ee8",
    "39e7b443-3c8e-5161-be90-b92af91ff7eb",
    "bff94d5c-03ab-5058-8b6e-505aa4a18d38",
    "67b49b24-5bae-50a1-a0e5-d6bd19531943",
    "19300fc4-c3bb-559b-abc6-1125e2e00876",
    "616449bf-357b-5f52-86bd-d767f6028645",
    "b602f2f1-8150-5dde-a3a9-25828d058cf4",
    "b599e259-f712-5b09-8ff6-dfc779110418",
    "bb44b19c-0f60-5609-a3e2-02901d10f761",
    "a3881177-4d9f-546a-aca8-a6684438fedd",
    "bea88144-79ee-58c2-9a23-57dd60dbdf8a",
    "a774300b-2f03-5bfc-b9a0-cf8e81b50fbb",
    "72c09e6c-ae34-58db-a9fe-a2e7ee8eceeb",
    "4cd52771-76b4-54ad-bae3-2ad863ffeedc",
    "ad3a9edc-19b5-5a02-a5f4-5d68081b3a34",
    "ad3a9edc-19b5-5a02-a5f4-5d68081b3a34",
    "ad3a9edc-19b5-5a02-a5f4-5d68081b3a34",
    "ad3a9edc-19b5-5a02-a5f4-5d68081b3a34",
    "7a78745c-e4ff-5d1f-abd0-39b5f517109b",
    "8eb7ea40-2bb6-5828-98af-5c8686c67a34",
    "f04f445f-dcd7-5044-8a7f-5d6f37a8e608",
    "ff5c534d-1dd9-5dac-be2e-b57f812f36ac",
    "784f054c-689a-52d4-b683-2c07a4c4e395",
    "c560f33c-96a0-59d3-9c03-cb4851aca53c",
    "d1bbb909-ad34-5e4a-98d3-e8b2d2e9cd50",
    "5dedff05-471c-5008-abe0-373d163d4c85",
    "cdc29dd5-3880-5993-a5da-620c7f7f5dab",
    "3faf342b-f569-586f-9c82-51a8622ec01f",
    "5b8be569-f3d2-5980-a844-8231a72bc283",
    "cd754304-5c6d-5724-b006-c2f0d205938a",
    "82a28177-6ff8-5b2b-987e-12dc2b9d09bd",
    "10a85105-5450-5237-9d56-d3ff8be27225",
    "50b41172-5655-58e4-8a89-aa4a3356f807",
    "efd9c026-048e-54c6-af45-61a38b92adfd",
    "f7b49d58-13e0-52f4-bcbd-15b3c3559491",
    "f3982f7e-3ea7-5abd-9212-298da881dcdd",
    "884205e8-6dd4-5034-91e8-c5a630c843ee",
    "4aa9e46a-37c2-531b-8803-adcac6bbd8bf",
    "7bb9fed4-a4b6-546a-9138-ad66b07df8b0",
    "b4766f93-156a-5000-9b60-a7d693dfe367",
    "91bf6157-840d-5cb9-ba67-ff1057be8e8d",
    "7b059d50-6ef5-5627-a246-77acd1693921",
    "7067912b-58e2-5c41-961d-546e00a566de",
    "0354892b-bf36-5f39-89ba-ae83abcc353b",
    "655a9aa2-ee49-592e-ab27-6b54893cc058",
    "dc7765b9-2583-5b20-9b9e-17d716ce515c",
    "a1893cd1-a227-516f-876c-cd17bf1c51b4",
    "82675ccc-691c-59ec-b4c7-da6abb479c02",
    "b935a385-2787-5093-a1a1-036598b3f410",
    "b13cbfb6-fda9-53de-87e3-b1ef0a0d6330",
    "6e9ed988-471d-5e3c-9f60-47521625a00b",
    "cb52ff80-955e-5641-97ab-7f96e18b2e0d",
    "661594ce-9cf9-56de-9fa4-6d1a1335ade7",
    "711c69d9-d111-519f-b3f2-b6edc77f2380",
    "f9120b90-a140-5709-b32c-e36f618febb6",
    "3454afb3-2e92-5bae-b43d-8ee3e9993ba3",
    "e719a581-271a-5368-88d9-aad40d4f4638",
    "6c9234a2-0700-5e30-aec2-71c4e7213649",
    "3afd7a93-150a-5c02-ad1d-889bfc58a563",
    "69396201-e74f-593c-8a39-d010c05b4b2b",
    "b0130cda-d934-51d5-880e-4f968d59e10b",
    "abe3eb03-d6e4-5302-b38f-0129513f5d8d",
    "3688bcd7-c0fa-53e8-৯b8a-12e7ba6cb971"
]

In [3]:
raw_data = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv")

In [4]:
raw_data

,ticket_uuid,attachment_id,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,s3_link,textract_job_id,textract_s3_link
0,0bb04c7d-e0a5-54c2-b9ef-b6ff6394e864,7a78745c-e4ff-5d1f-abd0-39b5f517109b,4070071 967072\nAntrag auf Erlass eines Pfändu...,01.02.2022/1643724050_Doc_01022022_000000001_1...,attachment_and_transfer_order,4070071 967072\nantrag auf erlass eines pfändu...,NaN,False,False,NaN,aa53a00a4be3004cf77a29bad626bfd877a85217942166...,s3://pair-data-engineering-new/ocr_prepared_ou...
1,722c7224-fa7c-5942-a4cf-c2ddb70c1e94,8eb7ea40-2bb6-5828-98af-5c8686c67a34,Antrag auf Erlass eines Pfänd 070071 967072\nÜ...,01.02.2022/1643724051_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfänd 070071 967072\nü...,NaN,False,False,NaN,a526cbb704ee1ce316ef92311493578034721ce8d1ac27...,s3://pair-data-engineering-new/ocr_prepared_ou...
2,16e88ab5-96dc-5afa-8447-7d068aacc6a7,dcf3e789-6aa7-595d-94c7-9133ae2b2375,Antrag auf Erlass eines Pfändun\n4070071 96707...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändun\n4070071 96707...,NaN,False,False,NaN,119c66b09111a9c414741acb6a520aceac59eb2c31deeb...,s3://pair-data-engineering-new/ocr_prepared_ou...
3,6f59a98b-d33d-5a0c-9bc4-05800e95c2b2,f04f445f-dcd7-5044-8a7f-5d6f37a8e608,Antrag auf Erlass eines Pfändungs\n4 070071 96...,01.02.2022/1643724052_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändungs\n4 070071 96...,NaN,False,False,NaN,e48ca3ce0673313350c06532a2dea4fe82acf6243da3f1...,s3://pair-data-engineering-new/ocr_prepared_ou...
4,19e6fa9c-6c9b-5701-a805-05b6bce007e8,ff5c534d-1dd9-5dac-be2e-b57f812f36ac,Antrag auf Erlass eines Pfändu\n4070071 967072...,01.02.2022/1643724053_Doc_01022022_000000001_1...,attachment_and_transfer_order,antrag auf erlass eines pfändu\n4070071 967072...,NaN,False,False,NaN,cb548acfe61af12fd9b1dd59419b673b34b7e32ff0bc1f...,s3://pair-data-engineering-new/ocr_prepared_ou...
...,...,...,...,...,...,...,...,...,...,...,...,...
5378,eae76ab8-0752-5c44-8726-5f920f8e6f0a,51170436,Gerichtsvollzieher\nSiemensstraße 20\nSteinsie...,NaN,fp_invoice,NaN,NaN,False,False,NaN,080d5f6c1c195c1abfcf9107e6bcf70367e6bd8c521df1...,s3://pair-data-engineering-new/ocr_prepared_ou...
5379,c6723ec9-b3bd-53f3-a7ba-121f8e6d708a,52618215,Wildeshauser Str. 21\nSandra Zenker\n26197 Ahl...,NaN,bailiff_ip,NaN,NaN,False,False,NaN,dcbb82c7982aa1a6f43b73f48cf71e520983dfeaf361f4...,s3://pair-data-engineering-new/ocr_prepared_ou...
5380,d66ee21e-705e-5729-9a82-d5821667f99d,50758910,André Rucks\nSittarder Str. 72\n52511 Geilenki...,NaN,tbd,NaN,NaN,False,False,NaN,e60667129a00c38ab5c8e70001a2ebbd49c13e9eb6e0f3...,s3://pair-data-engineering-new/ocr_prepared_ou...
5381,33751aae-53eb-5874-910a-65d80b66e082,51118089,Sonnenscheinpfad 28\n44879 Bochum\nTel. 0234/4...,NaN,fp_invoice,NaN,NaN,False,False,NaN,01a1a0d3674e331b26566e92f62e666c0a295ef7eca672...,s3://pair-data-engineering-new/ocr_prepared_ou...


## Split data into groups
- **FP (False Positive)**: `attachment_and_transfer_order` docs wrongly classified as `pfub_erlass`
- **Actual PFUB**: Documents where `is_pfub=True` (real pfub_erlass-type docs)

In [5]:
# Filter attachment_and_transfer_order documents
ato_docs = raw_data[raw_data["document_type"] == "attachment_and_transfer_order"]

# FP: wrongly classified as pfub_erlass
fp_docs = ato_docs[ato_docs["attachment_id"].isin(WRONG_ATTACHMENT_IDS)]

# Actual PFUB documents (is_pfub=True)
actual_pfub = raw_data[raw_data["is_pfub"] == True]

print(f"FP (attachment_and_transfer_order misclassified as pfub_erlass): {len(fp_docs)}")
print(f"Actual PFUB documents (is_pfub=True): {len(actual_pfub)}")
print(f"  - Doc types in actual PFUB: {actual_pfub['document_type'].value_counts().to_dict()}")

FP (attachment_and_transfer_order misclassified as pfub_erlass): 140
Actual PFUB documents (is_pfub=True): 471
  - Doc types in actual PFUB: {'approved_seizure': 406, 'approved_attachment_and_transfer_order': 64, 'court_inbox': 1}


## Keyword Extraction Helper

Tokenize and count word frequencies per group to find distinguishing keywords.

In [6]:
def tokenize(text):
    """Lowercase, split on non-alphanumeric, remove short tokens."""
    if pd.isna(text):
        return []
    tokens = re.findall(r'\b[a-zäöüß]{3,}\b', str(text).lower())
    return tokens

def get_document_freq(docs_series):
    """Return document frequency (in how many docs each word appears)."""
    counter = Counter()
    for text in docs_series:
        counter.update(set(tokenize(text)))
    return counter

# Get word frequencies for each group
fp_freq = get_document_freq(fp_docs["text"])
pfub_freq = get_document_freq(actual_pfub["text"])

print(f"Unique tokens - FP: {len(fp_freq)}, Actual PFUB: {len(pfub_freq)}")

Unique tokens - FP: 2141, Actual PFUB: 2103


## Shared keywords between FP and PFUB

Words that appear frequently in both FP docs and actual PFUB docs — these are the overlapping keywords that confuse the LLM.

In [7]:
# Compute document frequency ratios
all_words = set(fp_freq.keys()) | set(pfub_freq.keys())

fp_total = len(fp_docs)
pfub_total = len(actual_pfub)

rows = []
for word in all_words:
    fp_pct = fp_freq.get(word, 0) / fp_total * 100 if fp_total > 0 else 0
    pfub_pct = pfub_freq.get(word, 0) / pfub_total * 100 if pfub_total > 0 else 0
    rows.append({
        "word": word,
        "fp_doc_pct": round(fp_pct, 1),
        "pfub_doc_pct": round(pfub_pct, 1),
        "overlap": round(min(fp_pct, pfub_pct), 1),
    })

keyword_df = pd.DataFrame(rows)

# Shared keywords: appear frequently in both groups
shared_keywords = keyword_df[
    (keyword_df["fp_doc_pct"] > 10) & (keyword_df["pfub_doc_pct"] > 10)
].sort_values("overlap", ascending=False)

print("Shared keywords between FP and actual PFUB (confusing the LLM):")
print(f"{'Word':<30} {'FP %':>8} {'PFUB %':>8} {'Overlap':>8}")
print("-" * 56)
for _, row in shared_keywords.head(30).iterrows():
    print(f"{row['word']:<30} {row['fp_doc_pct']:>7.1f}% {row['pfub_doc_pct']:>7.1f}% {row['overlap']:>7.1f}%")

Shared keywords between FP and actual PFUB (confusing the LLM):
Word                               FP %   PFUB %  Overlap
--------------------------------------------------------
der                              100.0%   100.0%   100.0%
gmbh                             100.0%   100.0%   100.0%
und                              100.0%   100.0%   100.0%
amtsgericht                      100.0%   100.0%   100.0%
finance                          100.0%   100.0%   100.0%
pair                             100.0%   100.0%   100.0%
berlin                           100.0%   100.0%   100.0%
die                              100.0%    99.4%    99.4%
wurde                            100.0%    97.9%    97.9%
bitte                            100.0%    92.6%    92.6%
zustellung                       100.0%    91.1%    91.1%
mit                              100.0%    91.1%    91.1%
antrag                           100.0%    90.9%    90.9%
unterschrift                      95.7%    82.6%    82.6%
durch    

## Keywords distinctive to each group

Keywords that appear much more in one group than the other — use these to adjust the prompt.

In [9]:
# Keywords distinctive to FP (appear more in FP than PFUB)
fp_distinctive = keyword_df[
    (keyword_df["fp_doc_pct"] > 15) & ((keyword_df["fp_doc_pct"] - keyword_df["pfub_doc_pct"]) > 10)
].copy()
fp_distinctive["diff"] = fp_distinctive["fp_doc_pct"] - fp_distinctive["pfub_doc_pct"]
fp_distinctive = fp_distinctive.sort_values("diff", ascending=False)

print("Keywords distinctive to FP docs (should NOT trigger pfub_erlass):")
print(f"{'Word':<30} {'FP %':>8} {'PFUB %':>8} {'Diff':>8}")
print("-" * 56)
for _, row in fp_distinctive.head(30).iterrows():
    print(f"{row['word']:<30} {row['fp_doc_pct']:>7.1f}% {row['pfub_doc_pct']:>7.1f}% {row['diff']:>+7.1f}%")

print("\n" + "=" * 56 + "\n")

# Keywords distinctive to actual PFUB (appear more in PFUB than FP)
pfub_distinctive = keyword_df[
    (keyword_df["pfub_doc_pct"] > 15) & ((keyword_df["pfub_doc_pct"] - keyword_df["fp_doc_pct"]) > 10)
].copy()
pfub_distinctive["diff"] = pfub_distinctive["pfub_doc_pct"] - pfub_distinctive["fp_doc_pct"]
pfub_distinctive = pfub_distinctive.sort_values("diff", ascending=False)

print("Keywords distinctive to actual PFUB (true pfub_erlass signals):")
print(f"{'Word':<30} {'PFUB %':>8} {'FP %':>8} {'Diff':>8}")
print("-" * 56)
for _, row in pfub_distinctive.head(30).iterrows():
    print(f"{row['word']:<30} {row['pfub_doc_pct']:>7.1f}% {row['fp_doc_pct']:>7.1f}% {row['diff']:>+7.1f}%")

Keywords distinctive to FP docs (should NOT trigger pfub_erlass):
Word                               FP %   PFUB %     Diff
--------------------------------------------------------
verhältnisse                     100.0%     0.0%  +100.0%
schuldnervertreters              100.0%     0.0%  +100.0%
beginnt                          100.0%     0.0%  +100.0%
beantragt                        100.0%     0.0%  +100.0%
formular                         100.0%     0.0%  +100.0%
gewöhnlicher                     100.0%     0.0%  +100.0%
geldforderungen                  100.0%     0.0%  +100.0%
gläubigervertreters              100.0%     0.0%  +100.0%
zweckmäßige                      100.0%     0.0%  +100.0%
bewilligen                       100.0%     0.0%  +100.0%
vermitteln                       100.0%     0.0%  +100.0%
gläubiger                        100.0%     0.0%  +100.0%
geeignetes                       100.0%     0.0%  +100.0%
schaftlichen                     100.0%     0.0%  +100.0%
etc    

Keywords might help:

schuldnervertreters
gläubigervertreters
sozialleistungen
verrechnungsscheck
zivilprozessordnung

Sections:

- Aktenzeichen des Schuldnervertreters
- Aktenzeichen des Gläubigervertreter
- Zusammenrechnung von Arbeitseinkommen und Sozialleistungen
- Verrechnungsscheck für Gerichtskosten
